In [ ]:
import random
import math
from collections import defaultdict

random.seed(42)

# ============================================================
# SEQUENTIAL DECISION-MAKING AGENT
# Example domain: A rover that must collect samples over time
# while managing battery and uncertain weather.
# ============================================================

# ------------------------------------------------------------
# Part 1 + Part 2: Full-state MDP
# ------------------------------------------------------------

LOCATIONS = ["base", "field"]
BATTERIES = ["low", "high"]
WEATHERS = ["sunny", "dusty"]

STATES = [(loc, batt, weather) for loc in LOCATIONS for batt in BATTERIES for weather in WEATHERS]
ACTIONS = ["charge", "go_field", "sample", "return_base"]

DISCOUNT = 0.92

def available_actions(state):
    loc, batt, weather = state
    acts = []
    if loc == "base":
        acts.append("charge")
        if batt == "high":
            acts.append("go_field")
    if loc == "field":
        acts.append("sample")
        acts.append("return_base")
    return acts

def weather_transition_probs(current_weather):
    # Weather evolves stochastically over time.
    if current_weather == "sunny":
        return {"sunny": 0.75, "dusty": 0.25}
    else:
        return {"sunny": 0.35, "dusty": 0.65}

def transition_reward(state, action):
    """
    Returns list of (probability, next_state, reward)
    """
    loc, batt, weather = state
    outcomes = []

    if action not in available_actions(state):
        return [(1.0, state, -50.0)]  # invalid action penalty

    next_weathers = weather_transition_probs(weather)

    if action == "charge":
        # At base, charging gives small cost but makes battery high next step
        for nw, p in next_weathers.items():
            ns = ("base", "high", nw)
            reward = -2.0
            outcomes.append((p, ns, reward))

    elif action == "go_field":
        # Must be at base with high battery.
        # Travel is riskier in dusty weather.
        for nw, p_nw in next_weathers.items():
            if weather == "sunny":
                success_prob = 0.90
            else:
                success_prob = 0.65

            # Success: rover reaches field, battery becomes low
            outcomes.append((p_nw * success_prob, ("field", "low", nw), -4.0))

            # Failure: rover remains at base, still high battery, travel wasted
            outcomes.append((p_nw * (1 - success_prob), ("base", "high", nw), -10.0))

    elif action == "sample":
        # Sampling at field. Better reward in sunny weather, worse in dusty weather.
        # Battery stays low at field, so repeated sampling is dangerous but possible.
        for nw, p_nw in next_weathers.items():
            if weather == "sunny":
                reward = 16.0
            else:
                reward = 5.0
            outcomes.append((p_nw, ("field", "low", nw), reward))

    elif action == "return_base":
        # Returning to base is safer than going out, but still costs time/energy.
        for nw, p_nw in next_weathers.items():
            if weather == "sunny":
                success_prob = 0.95
            else:
                success_prob = 0.80

            outcomes.append((p_nw * success_prob, ("base", "low", nw), -3.0))
            outcomes.append((p_nw * (1 - success_prob), ("field", "low", nw), -8.0))

    return outcomes

def expected_action_value(state, action, V, gamma=DISCOUNT):
    total = 0.0
    for prob, next_state, reward in transition_reward(state, action):
        total += prob * (reward + gamma * V[next_state])
    return total

def value_iteration(states, gamma=DISCOUNT, theta=1e-8, max_iters=1000):
    V = {s: 0.0 for s in states}
    for _ in range(max_iters):
        delta = 0.0
        newV = V.copy()
        for s in states:
            acts = available_actions(s)
            if not acts:
                continue
            best = max(expected_action_value(s, a, V, gamma) for a in acts)
            newV[s] = best
            delta = max(delta, abs(newV[s] - V[s]))
        V = newV
        if delta < theta:
            break

    policy = {}
    Q = {}
    for s in states:
        acts = available_actions(s)
        if acts:
            action_values = {a: expected_action_value(s, a, V, gamma) for a in acts}
            Q[s] = action_values
            policy[s] = max(action_values, key=action_values.get)
    return V, policy, Q

V_star, optimal_policy, Q_star = value_iteration(STATES)

# Greedy-myopic policy using only immediate expected reward
def greedy_immediate_action(state):
    acts = available_actions(state)
    scores = {}
    for a in acts:
        total = 0.0
        for prob, next_state, reward in transition_reward(state, a):
            total += prob * reward
        scores[a] = total
    return max(scores, key=scores.get), scores

# ------------------------------------------------------------
# Part 3: Multi-Armed Bandit
# ------------------------------------------------------------

class BernoulliBandit:
    def __init__(self, probs):
        self.probs = probs
        self.k = len(probs)

    def pull(self, arm):
        return 1 if random.random() < self.probs[arm] else 0

def run_pure_greedy_bandit(bandit, steps=300):
    counts = [0] * bandit.k
    values = [0.0] * bandit.k
    rewards = []

    # Initialize by trying each arm once
    for arm in range(bandit.k):
        r = bandit.pull(arm)
        counts[arm] += 1
        values[arm] += (r - values[arm]) / counts[arm]
        rewards.append(r)

    for _ in range(steps - bandit.k):
        arm = max(range(bandit.k), key=lambda a: values[a])
        r = bandit.pull(arm)
        counts[arm] += 1
        values[arm] += (r - values[arm]) / counts[arm]
        rewards.append(r)

    return {
        "counts": counts,
        "values": values,
        "total_reward": sum(rewards),
        "average_reward": sum(rewards) / len(rewards),
    }

def run_epsilon_greedy_bandit(bandit, epsilon=0.10, steps=300):
    counts = [0] * bandit.k
    values = [0.0] * bandit.k
    rewards = []

    for arm in range(bandit.k):
        r = bandit.pull(arm)
        counts[arm] += 1
        values[arm] += (r - values[arm]) / counts[arm]
        rewards.append(r)

    for _ in range(steps - bandit.k):
        if random.random() < epsilon:
            arm = random.randrange(bandit.k)
        else:
            arm = max(range(bandit.k), key=lambda a: values[a])

        r = bandit.pull(arm)
        counts[arm] += 1
        values[arm] += (r - values[arm]) / counts[arm]
        rewards.append(r)

    return {
        "counts": counts,
        "values": values,
        "total_reward": sum(rewards),
        "average_reward": sum(rewards) / len(rewards),
    }

def run_ucb1_bandit(bandit, steps=300):
    counts = [0] * bandit.k
    values = [0.0] * bandit.k
    rewards = []

    for arm in range(bandit.k):
        r = bandit.pull(arm)
        counts[arm] += 1
        values[arm] += (r - values[arm]) / counts[arm]
        rewards.append(r)

    for t in range(bandit.k, steps):
        arm = max(
            range(bandit.k),
            key=lambda a: values[a] + math.sqrt((2 * math.log(t + 1)) / counts[a])
        )
        r = bandit.pull(arm)
        counts[arm] += 1
        values[arm] += (r - values[arm]) / counts[arm]
        rewards.append(r)

    return {
        "counts": counts,
        "values": values,
        "total_reward": sum(rewards),
        "average_reward": sum(rewards) / len(rewards),
    }

# ------------------------------------------------------------
# Part 4 + Part 5: Partial Observability / Approximate POMDP
# ------------------------------------------------------------

OBSERVATIONS = ["clear_sensor", "dust_alert"]

def observation_probs(true_weather):
    # Noisy sensor: more likely to report clear in sunny weather,
    # more likely to report dust alert in dusty weather.
    if true_weather == "sunny":
        return {"clear_sensor": 0.80, "dust_alert": 0.20}
    else:
        return {"clear_sensor": 0.25, "dust_alert": 0.75}

def normalize(dist):
    total = sum(dist.values())
    return {k: v / total for k, v in dist.items()}

def weather_belief_predict(belief):
    new_belief = {"sunny": 0.0, "dusty": 0.0}
    for w in WEATHERS:
        for nw, p in weather_transition_probs(w).items():
            new_belief[nw] += belief[w] * p
    return normalize(new_belief)

def weather_belief_update(predicted_belief, observation):
    posterior = {}
    for w in WEATHERS:
        posterior[w] = observation_probs(w)[observation] * predicted_belief[w]
    return normalize(posterior)

def belief_expected_immediate_reward(loc, batt, belief, action):
    total = 0.0
    for w in WEATHERS:
        state = (loc, batt, w)
        # expected immediate reward conditioned on a fixed weather
        r = 0.0
        for prob, next_state, reward in transition_reward(state, action):
            r += prob * reward
        total += belief[w] * r
    return total

def belief_transition_distribution(loc, batt, belief, action):
    """
    From known location+battery and weather belief, compute next distribution over
    (loc, batt, weather). Used for approximate planning under uncertainty.
    """
    dist = defaultdict(float)
    for w in WEATHERS:
        s = (loc, batt, w)
        for p, ns, r in transition_reward(s, action):
            dist[ns] += belief[w] * p
    return dict(dist)

def approximate_belief_value(loc, batt, belief, depth=3, gamma=DISCOUNT):
    """
    Approximate value for a belief state using finite-horizon lookahead.
    Exact POMDP solving is avoided because the belief space is continuous.
    """
    acts = available_actions((loc, batt, "sunny"))  # actions depend only on loc+batt here
    if depth == 0 or not acts:
        return 0.0

    best = -float("inf")
    for action in acts:
        immediate = belief_expected_immediate_reward(loc, batt, belief, action)
        next_dist = belief_transition_distribution(loc, batt, belief, action)

        # Approximate future value by collapsing next distribution into:
        # known loc+batt marginal + predicted weather belief
        future_value = 0.0
        for (nloc, nbatt, nw), prob_state in next_dist.items():
            predicted_weather_belief = weather_belief_predict(belief)
            future_value += prob_state * approximate_belief_value(
                nloc, nbatt, predicted_weather_belief, depth - 1, gamma
            )

        total = immediate + gamma * future_value
        best = max(best, total)
    return best

def choose_action_from_belief(loc, batt, belief, depth=3, gamma=DISCOUNT):
    acts = available_actions((loc, batt, "sunny"))
    scores = {}
    for action in acts:
        immediate = belief_expected_immediate_reward(loc, batt, belief, action)
        next_dist = belief_transition_distribution(loc, batt, belief, action)

        future_value = 0.0
        for (nloc, nbatt, nw), prob_state in next_dist.items():
            predicted_weather_belief = weather_belief_predict(belief)
            future_value += prob_state * approximate_belief_value(
                nloc, nbatt, predicted_weather_belief, depth - 1, gamma
            )

        scores[action] = immediate + gamma * future_value
    best_action = max(scores, key=scores.get)
    return best_action, scores

def sample_observation(true_weather):
    probs = observation_probs(true_weather)
    x = random.random()
    cumulative = 0.0
    for obs, p in probs.items():
        cumulative += p
        if x <= cumulative:
            return obs
    return "dust_alert"

# ------------------------------------------------------------
# Reporting
# ------------------------------------------------------------

print("=" * 78)
print("SEQUENTIAL DECISION-MAKING AGENT: ROVER DOMAIN")
print("=" * 78)

print("\nPART 1: SEQUENTIAL DECISION PROBLEM")
print("States: (location, battery, weather)")
print("Locations:", LOCATIONS)
print("Battery levels:", BATTERIES)
print("Weather states:", WEATHERS)
print("Actions:", ACTIONS)
print("\nWhy greedy decisions can fail:")
print("- Sampling in the field gives immediate reward.")
print("- But if the rover ignores future consequences, it may stay in a low-battery state")
print("  and be stuck in bad weather, reducing long-term return.")
print("- Returning to base or charging can be better long-term even if they look worse immediately.")

example_state = ("field", "low", "sunny")
greedy_action, greedy_scores = greedy_immediate_action(example_state)

print("\nExample state for comparison:", example_state)
print("Immediate-only expected rewards:")
for a, val in greedy_scores.items():
    print(f"  {a:12s} -> {val:8.3f}")
print("Greedy immediate action:", greedy_action)
print("Optimal long-term action from value iteration:", optimal_policy[example_state])

print("\nPART 2: VALUE ITERATION RESULTS")
print("Discount factor:", DISCOUNT)
print("\nOptimal state values and policy:")
for s in STATES:
    print(f"State {s}: V* = {V_star[s]:8.3f}, policy = {optimal_policy[s]}")
    for a, q in Q_star[s].items():
        print(f"    Q({a}) = {q:8.3f}")

print("\nPART 3: MULTI-ARMED BANDIT")
print("Bandit version of the problem:")
print("- Arm 0 = sample at Site A")
print("- Arm 1 = sample at Site B")
print("- True rewards are unknown initially.")

true_site_probs = [0.45, 0.68]
bandit = BernoulliBandit(true_site_probs)

greedy_result = run_pure_greedy_bandit(bandit, steps=300)
bandit = BernoulliBandit(true_site_probs)
eps_result = run_epsilon_greedy_bandit(bandit, epsilon=0.10, steps=300)
bandit = BernoulliBandit(true_site_probs)
ucb_result = run_ucb1_bandit(bandit, steps=300)

print("\nTrue success probabilities:", true_site_probs)
print("\nPure Greedy:")
print("  counts =", greedy_result["counts"])
print("  estimated values =", [round(v, 3) for v in greedy_result["values"]])
print("  total reward =", greedy_result["total_reward"])
print("  average reward =", round(greedy_result["average_reward"], 3))

print("\nEpsilon-Greedy (epsilon=0.10):")
print("  counts =", eps_result["counts"])
print("  estimated values =", [round(v, 3) for v in eps_result["values"]])
print("  total reward =", eps_result["total_reward"])
print("  average reward =", round(eps_result["average_reward"], 3))

print("\nUCB1:")
print("  counts =", ucb_result["counts"])
print("  estimated values =", [round(v, 3) for v in ucb_result["values"]])
print("  total reward =", ucb_result["total_reward"])
print("  average reward =", round(ucb_result["average_reward"], 3))

print("\nPART 4: PARTIAL OBSERVABILITY")
print("Now weather is hidden. The rover knows only location and battery.")
print("It maintains a belief over weather and updates it from noisy observations.")

initial_belief = {"sunny": 0.50, "dusty": 0.50}
print("\nInitial belief:", initial_belief)

true_weather_demo = "dusty"
observation = sample_observation(true_weather_demo)
predicted = weather_belief_predict(initial_belief)
posterior = weather_belief_update(predicted, observation)

print("Suppose the true hidden weather for this step is:", true_weather_demo)
print("Noisy observation received:", observation)
print("Predicted belief after weather dynamics:", {k: round(v, 3) for k, v in predicted.items()})
print("Posterior belief after observation:", {k: round(v, 3) for k, v in posterior.items()})

print("\nPART 5: APPROXIMATE POMDP DECISION")
print("Approximation used: finite-horizon lookahead over belief states.")
print("Reason exact solutions are infeasible: the belief state is continuous,")
print("so exact dynamic programming over all beliefs is computationally expensive.")

belief_state_loc = "base"
belief_state_batt = "high"
belief_action, belief_scores = choose_action_from_belief(
    belief_state_loc, belief_state_batt, posterior, depth=3, gamma=DISCOUNT
)

print("\nBelief-state decision example")
print("Known location:", belief_state_loc)
print("Known battery:", belief_state_batt)
print("Belief over weather:", {k: round(v, 3) for k, v in posterior.items()})
for a, score in belief_scores.items():
    print(f"  Approx belief value for {a:12s} -> {score:8.3f}")
print("Chosen action from belief state:", belief_action)

print("\nSUMMARY")
print("- Value iteration computed optimal long-term values and policy for the full-state MDP.")
print("- Bandit strategies handled unknown rewards by balancing exploration and exploitation.")
print("- Belief updates handled hidden weather under partial observability.")
print("- A finite-horizon belief planner approximated action choice in the POMDP setting.")
print("=" * 78)

## Questions

### How did planning over time change decisions?

Planning over time changed decisions by making the agent care about future consequences rather than only immediate rewards. In the MDP setting, an action was evaluated not just by what it produced right away, but also by how it affected later states such as future location, remaining energy, and readiness for the important 18:00 practice objective.

This created situations where the optimal policy chose actions that looked worse in the short term but led to better outcomes later. For example, resting earlier could be better than moving immediately if it preserved enough energy to arrive at VillagePark in a better condition at the practice time. Similarly, moving toward a strategically useful location earlier could be better than choosing a locally attractive option that left the agent poorly positioned later.

In this way, planning over time transformed the problem from “What gives the best immediate reward?” into “What sequence of actions leads to the best overall return?” That is exactly why the optimal MDP policy could differ from a greedy strategy.

### When did exploration improve performance?

Exploration improved performance in the bandit setting when the agent did not yet know the true average reward of each option. At the beginning, all reward estimates were uncertain, so committing too early to one arm could be risky.

A purely greedy strategy could get stuck exploiting an arm that only looked good because of favorable early noise. By contrast, ε-greedy improved performance by occasionally trying other actions, which gave the agent a chance to discover better options and correct bad initial estimates. UCB1 improved performance in an even more focused way by favoring arms with high uncertainty, balancing what looked promising with what still needed more evidence.

So exploration was most useful early in learning and whenever uncertainty about arm quality remained high. Once the best arm became clear, exploitation mattered more. But without enough exploration first, the agent could easily settle on a suboptimal action.

### How did partial observability affect strategy?

Partial observability changed the strategy by forcing the agent to reason about probabilities instead of directly observed facts. In the POMDP section, the crowd status of StudentCenter was hidden, so the agent could no longer condition its action on the true state. Instead, it had to maintain a belief \( b_t = P(Busy) \).

Because the hidden state evolved over time and observations were noisy, the agent also had to update that belief dynamically using a Bayes filter. This made the strategy more cautious and information-sensitive. Rather than simply asking “Is StudentCenter busy?”, the agent had to ask “How likely is StudentCenter to be busy, and is it worth paying to learn more before acting?”

This also introduced a new type of action: the information-gathering action `CHECK_SC`. The value of that action depended on whether the expected improvement in future decision quality was large enough to justify the sensing cost. So partial observability changed the strategy by making belief management and information gathering part of the decision process itself.

### What trade-offs did you make using approximate POMDP solutions?

The main trade-off in using approximate POMDP solutions was between computational tractability and exact optimality.

Exact POMDP solving is difficult because the belief state is continuous and the number of possible future belief trajectories grows very quickly. To avoid that explosion in complexity, this notebook used Monte Carlo rollouts over a limited horizon. That made the problem feasible to solve and still allowed the model to account for hidden state, noisy observations, and discounted future rewards.

However, that approximation came with limitations:
- the solution depends on sampled rollouts rather than exact evaluation,
- action values may contain simulation noise,
- the planning horizon is short, so very long-term effects may be underrepresented,
- and the quality of the result depends on the default rollout policy used after the first action.

So the trade-off was clear: the approximate approach was practical and flexible, but it sacrificed some precision and optimality. It is a reasonable choice when exact POMDP methods are too expensive, especially for demonstrating the main ideas of belief-based decision-making.

## Part 5: Solving POMDPs (17.5) — Reflection

### What approximation you used
I used a **finite-horizon belief-state lookahead** approximation. Instead of solving the full POMDP exactly, the agent evaluates actions by:
- Computing expected rewards under the current belief distribution
- Simulating a limited number of future steps (depth-limited planning)
- Approximating future value recursively based on predicted belief updates

This avoids operating over the full continuous belief space.

---

### Why exact solutions are infeasible
Exact POMDP solutions are infeasible because:
- The **belief space is continuous**, not discrete
- The number of possible belief states grows infinitely
- Value iteration over beliefs becomes computationally intractable
- The complexity increases exponentially with planning horizon and state size

Even small problems become too expensive to solve exactly.

---

### How did planning over time change decisions?
Planning over time made the agent:
- Prefer **long-term stability** over short-term reward
- Choose actions like returning or conserving resources instead of repeatedly exploiting immediate gains
- Anticipate future states (e.g., low battery or bad conditions) and act proactively

Without planning, the agent would behave myopically and perform worse over time.

---

### When did exploration improve performance?
Exploration improved performance when:
- The agent had **uncertainty about rewards or outcomes**
- Early information about actions was incomplete or misleading
- Trying less-selected actions revealed better long-term options

In the bandit setting, strategies like epsilon-greedy and UCB outperformed pure greedy because they avoided getting stuck in suboptimal choices.

---

### How did partial observability affect strategy?
Partial observability forced the agent to:
- Maintain a **belief distribution** instead of relying on exact states
- Make decisions under uncertainty about the true environment
- Use **probabilistic reasoning** and observations to update beliefs

This made the strategy more cautious and adaptive, since actions depended on estimated conditions rather than known ones.

---

### What trade-offs did you make using approximate POMDP solutions?
The main trade-offs were:

- **Accuracy vs efficiency**  
  The approximation sacrifices optimality for computational tractability.

- **Limited planning depth vs long-term optimality**  
  Finite-horizon lookahead may miss better long-term strategies.

- **Simplified belief updates vs full modeling**  
  The model does not fully capture all possible belief transitions.

- **Faster decision-making vs guaranteed optimal solutions**  
  The agent can act quickly, but cannot guarantee globally optimal behavior.

Overall, the approximation provides a practical balance between performance and computational feasibility.